<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 15 · 让服务在后台持续沉淀知识

前面的提取由我们手动触发。这次让 Server 自己按计划处理工作材料：普通材料进入 Memory 提取，带 Task Outcome 的完成记录进入 Experience 孵化，生成的候选仍等待审核。

需要真实 Generation 模型。为了直观看到周期，我们使用几秒的教学间隔；生产环境应根据吞吐和模型成本配置。每次运行请使用独立实验数据库，避免本篇后台任务处理其他实验的材料。

路线：保存普通材料与任务结果 → 开启调度 → 等待可观察结果 → 重启 → 检查没有重复候选。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("15", features=("generation",))
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 15", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 先写材料，保持调度关闭

我们执行一个真正的 Decimal 检查，把结果作为 Task Outcome 的证据。普通项目约定用于 Memory，任务完成记录用于 Experience；两条处理路径有各自的游标。

In [ ]:
from decimal import Decimal

from powercontext.http import (
    CaptureContentSourceRequest,
    CreateSourceRequest,
    ListArtifactCandidatesRequest,
    ListMemoryEntriesRequest,
)

assert Decimal("1.999") * 100 != (Decimal("1.999") * 100).to_integral_value()
rule = await client.create_source(
    scope_id,
    CreateSourceRequest(content="项目长期约定 amount：存储单位是整数分；拒绝负数、非有限值和超过两位小数的输入。"),
)
outcome = await client.capture_content_source(
    CaptureContentSourceRequest(
        scope_id=scope_id,
        source_id=f"{lab.run_id}:outcome",
        content="任务：修复 CSV 金额精度校验。执行 Python Decimal 检查，1.999 * 100 不是整数，验证应该拒绝该输入。结果：边界检查通过。经验：在整数转换之前校验精度，避免静默截断。",
        metadata={"kind": "task-outcome"},
    )
)
assert not (await client.list_artifact_candidates(ListArtifactCandidatesRequest(scope_id=scope_id))).candidates
print("材料已保存，尚无后台候选。")

## 开启两个周期，等待实际结果

轮询只是查看公开结果，不调用 flush 或 generate。截止时间到了仍没有结果，就让本篇失败并保留材料，不能把等待超时当作成功。

In [ ]:
import asyncio
import time

from powercontext.builtin.runtime.config import RuntimeConfig

lab.settings_overrides["runtime"] = RuntimeConfig(schedule_seconds=3, experience_schedule_seconds=4)
await lab.restart()
client = lab.client
deadline = time.monotonic() + 180
while True:
    memory = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
    inbox = await client.list_artifact_candidates(ListArtifactCandidatesRequest(scope_id=scope_id))
    if memory.entries and inbox.candidates:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("后台处理未在 180 秒内形成 Memory 和 Experience 候选，请检查模型与服务日志")
    await asyncio.sleep(2)
assert all(item.status == "pending" for item in inbox.candidates)
assert all(item.family == "experience" for item in inbox.candidates)
show({"Memory 条数": len(memory.entries), "待审 Experience 数": len(inbox.candidates)})
show(inbox.candidates[0].proposal)

## 重启后，不重复孵化已经处理过的窗口

游标与候选持久化之后，重启会继续处理新材料。我们等待两个完整周期，比较精确的候选 ID 集合。调度不会替人批准，也不会自动发布 Skill。

In [ ]:
candidate_ids = {item.candidate_id for item in inbox.candidates}
await lab.restart()
client = lab.client
await asyncio.sleep(10)
after = await client.list_artifact_candidates(ListArtifactCandidatesRequest(scope_id=scope_id))
assert {item.candidate_id for item in after.candidates} == candidate_ids
assert all(item.status == "pending" and item.result_artifact is None for item in after.candidates)
print("重启后候选集合保持一致；所有候选仍需审核。")

## 练习与验收

增加一条新的 task-outcome，再等待后台产生新候选。普通聊天材料不会因此自动变成 Experience。模型故障与恢复的实验见第 21 篇；这里不通过轮询主动触发生成。

接下来阅读 [16_skill_feedback_loop.ipynb](16_skill_feedback_loop.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")